# Performance: Measure First, Optimize Second
Performance work gets better the moment we stop guessing. Python performance is shaped by object allocation, lookups, protocol overhead, and algorithm choices, so careful measurement has to come before clever changes.

When people struggle with **Performance: Measure First, Optimize Second**, the struggle is often not about syntax. It is usually about trying to reason from surface appearance alone. Python rewards a deeper habit: do not ask only what the code *looks like*; ask what objects exist, what names or attributes point to them, what bytecode-level actions are likely happening, and which runtime protocol is being used.

That habit is what turns memorized rules into real understanding. It also makes interviews easier, because you stop giving slogan answers and start giving mechanism-based answers.


- Develop a measurement-first mindset
- Recognize the importance of algorithmic choices
- Use basic timing tools
- Avoid premature optimization


Performance is often about how much work is done, how much data is moved, and how many objects are created or revisited. The memory picture matters because unnecessary object creation and poor data structure choices can create real cost.


In [1]:
nums = list(range(10000))
lookup_list = nums
lookup_set = set(nums)
print(len(lookup_list), len(lookup_set))


10000 10000


Bytecode can explain some low-level differences, but the most important lesson here is scale: fewer operations and better data structures usually beat tiny instruction-level tricks.


In [2]:
import dis

def use_loop(values):
    total = 0
    for value in values:
        total += value
    return total

dis.dis(use_loop)


  3           0 RESUME                   0

  4           2 LOAD_CONST               1 (0)
              4 STORE_FAST               1 (total)

  5           6 LOAD_FAST                0 (values)
              8 GET_ITER
        >>   10 FOR_ITER                 7 (to 26)
             12 STORE_FAST               2 (value)

  6          14 LOAD_FAST                1 (total)
             16 LOAD_FAST                2 (value)
             18 BINARY_OP               13 (+=)
             22 STORE_FAST               1 (total)
             24 JUMP_BACKWARD            8 (to 10)

  7     >>   26 LOAD_FAST                1 (total)
             28 RETURN_VALUE


Human intuition is not a reliable profiler.

A set versus a list can change the cost profile of membership checks immediately.

Big-O is not everything, but it often matters more than micro-optimizing syntax.

Clarity is a performance feature for teams.


This is a much healthier habit than eyeballing speed by feel.


In [3]:
import timeit
print(timeit.timeit("sum(range(100))", number=10000))


0.005934700006037019


This is a classic case where the data structure choice matters a lot.


In [4]:
values = list(range(10000))
values_set = set(values)
print(9999 in values)
print(9999 in values_set)


True
True


Sometimes the easiest optimization is to do less.


In [5]:
names = ["Ada", "Grace", "Linus"]
upper_once = [name.upper() for name in names]
print(upper_once)


['ADA', 'GRACE', 'LINUS']


This is not only a classroom topic. **Performance: Measure First, Optimize Second** usually appears in real projects as a debugging problem, a design choice, a performance tradeoff, or a readability decision. In other words, this topic matters most when the codebase gets large enough that vague intuition stops being reliable.

That is why the low-level view is useful. It gives you something firmer than guesswork when code starts behaving in surprising ways.


- Optimizing before measuring
- Arguing about tiny speed wins while ignoring algorithm choice
- Sacrificing readability for unproven gains


- Why is premature optimization risky?
- What kinds of improvements tend to matter most?
- Why should you profile first?


- Measure first.
- Data structures and algorithms matter a lot.
- Readable code and performance can coexist.


If this notebook did its job, **Performance: Measure First, Optimize Second** should now feel less like a bag of special rules and more like a natural consequence of Python's runtime model. That is the standard we want: not just recall, but a calm explanation of what the interpreter is seeing and why the behavior follows from that.


A better way to understand Performance: Measure First, Optimize Second is to keep moving back and forth between three views of the same code.

The first view is the human view. This is what the code is trying to say. The second view is the object view. In that view, we ask which Python objects exist, which names refer to them, and which objects are being created, reused, mutated, or discarded. The third view is the interpreter view. In that view, we ask what protocol is being used, what bytecode is being executed, and which runtime rules are controlling the result.

Many learners stop after the first view. That is enough to copy code, but it is usually not enough to explain surprising behavior. Deep understanding starts when these three views become one mental movement instead of three unrelated topics.


There is also a fourth habit that helps a lot: failure-based thinking.

For Performance: Measure First, Optimize Second, ask yourself what normally goes wrong when the idea is only half-understood. Does someone confuse identity with equality? Do they expect copying where only rebinding happened? Do they expect a fresh iterator when they really have an exhausted one? Do they trust a default value that is secretly shared? Do they think a class attribute is per-instance state? Once you can predict the common failure modes before they happen, your understanding becomes much more stable.


In [6]:
import dis
import inspect

def probe_runtime(value):
    local_copy = value
    frame = inspect.currentframe()
    print('locals now:', frame.f_locals)
    return local_copy

print(probe_runtime('demo'))
print('\nbytecode for probe_runtime:')
dis.dis(probe_runtime)


locals now: {'value': 'demo', 'local_copy': 'demo', 'frame': <frame at 0x00000193CE593DC0, file 'C:\\Users\\VIHAAN\\AppData\\Local\\Temp\\ipykernel_5172\\939914315.py', line 7, code probe_runtime>}
demo

bytecode for probe_runtime:
  4           0 RESUME                   0

  5           2 LOAD_FAST                0 (value)
              4 STORE_FAST               1 (local_copy)

  6           6 LOAD_GLOBAL              0 (inspect)
             18 LOAD_METHOD              1 (currentframe)
             40 PRECALL                  0
             44 CALL                     0
             54 STORE_FAST               2 (frame)

  7          56 LOAD_GLOBAL              5 (NULL + print)
             68 LOAD_CONST               1 ('locals now:')
             70 LOAD_FAST                2 (frame)
             72 LOAD_ATTR                3 (f_locals)
             82 PRECALL                  2
             86 CALL                     2
             96 POP_TOP

  8          98 LOAD_FAST         

The deepest performance improvements usually come from understanding where the interpreter is spending effort: object creation, repeated lookups, Python-level loops, expensive conversions, or poor data-structure fit. Timing matters most when it is attached to that runtime story.


In [7]:
import sys
nums = list(range(5))
doubled = [n * 2 for n in nums]
print('original list size:', sys.getsizeof(nums))
print('new list size:', sys.getsizeof(doubled))


original list size: 104
new list size: 120


This topic becomes much more useful when you tie performance back to actual runtime causes: too many objects, too many lookups, too much Python-level looping, or the wrong data structure for the job. Measuring is the beginning, not the end. The deeper skill is connecting what you measured to what the interpreter is really doing.


## Final Takeaway

The deepest way to revise **Performance: Measure First, Optimize Second** is to stop treating it as a collection of syntax facts and instead treat it as a runtime story. Ask what objects are involved, what names or attributes point to those objects, what frame is active, and what protocol or bytecode path Python is following. If you can explain this topic at those four levels, then you understand much more than the visible syntax.

In interview terms, the strongest answer is usually not the shortest definition. It is a calm explanation of what Python is doing under the surface and why the behavior follows from that model. For revision, come back to this notebook and ask yourself: could I explain this entire chapter with one small code example, one memory-level explanation, and one interpreter-level explanation? If yes, the topic is becoming yours.
